In [1]:
import os
import glob
import pandas as pd
import numpy as np
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam


In [49]:
def csv2df(dir_path):
    train_data_frames, test_data_frames = [], []
    testing_files = ["continuous", "flooding", "normal", "plateau", "playback", "suppress"]
    csv_path_train = dir_path + "/train_1.csv" #first one is structured different than the rest
    df_temp = pd.read_csv(csv_path_train)
    train_data_frames.append(df_temp)
    for i in range(2, 5):
        csv_path_train = f"{dir_path}/train_{i}.csv"
        df_temp = pd.read_csv(csv_path_train, header = None, names=["Label",  "Time", "ID", "Signal1",  "Signal2",  "Signal3",  "Signal4"],
                                  dtype={"Label": "int8", "Time": "float32", "ID": "str",
           "Signal1": "float32", "Signal2": "float32",
           "Signal3": "float32", "Signal4": "float32"})
        train_data_frames.append(df_temp)
    for file_suffix in testing_files:
        csv_path_test = f"{dir_path}/test_{file_suffix}.csv"
        df_temp = pd.read_csv(csv_path_test, low_memory=False,header=0,  names=["Label",  "Time", "ID", "Signal1",  "Signal2",  "Signal3",  "Signal4"],
            dtype={"Label": "int8", "Time": "float32", "ID": "str",
           "Signal1": "float32", "Signal2": "float32",
           "Signal3": "float32", "Signal4": "float32"})
        test_data_frames.append(df_temp)
    test_df = pd.concat(test_data_frames, ignore_index=True)
    test_df = test_df[test_df["Label"].isin([0,1])]
    train_df = pd.concat(train_data_frames, ignore_index=True)

    # test_data.columns = train_data.columns #make sure that both dfs have the same column names 
    return train_df, test_df

train_data, test_data = csv2df("./SynCAN-DATA")


In [50]:
data = pd.concat([train_data, test_data], ignore_index=True)
# Split into features and labels
X = data.drop(columns=["Label"])
y = data["Label"]
# undersample normal traffic
normal = data[data["Label"] == 0]
attack = data[data["Label"] == 1]

In [ ]:
def combine_balance_split(train_data, test_data):
    # Combine train + test
    data = pd.concat([train_data, test_data], ignore_index=True)
    # Split into features and labels
    X = data.drop(columns=["Label"])
    y = data["Label"]
    # undersample normal traffic
    normal = data[data["Label"] == 0]
    attack = data[data["Label"] == 1]
    # Match counts
    normal_downsampled = resample(normal, #ERROR HERE 
                                replace=False,
                                n_samples=len(attack),
                                random_state=42)
    balanced = pd.concat([normal_downsampled, attack])
    balanced.interpolate(method='linear', inplace=True)
    # Forward fill any remaining NaNs (e.g., at start)
    balanced = balanced.ffill
    # Backward fill any remaining NaNs (just in case)
    balanced = balanced.bfill()
    X_bal = balanced.drop(columns=["Label"])
    y_bal = balanced["Label"]
    X_train, X_val, y_train, y_val = train_test_split(
        X_bal, y_bal, test_size=0.2, random_state=42, stratify=y_bal
    )
    return X_train, X_val, y_train, y_val
# X_train, X_val, y_train, y_val = combine_balance_split(train_data, test_data)

In [66]:
def create_windows(X, y, window_size=64):
    X_windows, y_windows = [], []
    for i in range(len(X) - window_size):
        X_windows.append(X.iloc[i:i+window_size].values)
        y_windows.append(y.iloc[i+window_size-1])  # last label in window
    return np.array(X_windows), np.array(y_windows)

In [67]:
def main():
    # Load all training (nromal) and test (normal + attack) CSVs 
    train_data, test_data = csv2df("./SynCAN-DATA")
    X_train, X_val, y_train, y_val = combine_balance_split(train_data, test_data)
    window_size = 64
    X_train, y_train = create_windows(X_train, y_train)
    X_train_w, y_train_w = create_windows(X_train, y_train, window_size)
    X_val_w, y_val_w = create_windows(X_val, y_val, window_size)
    np.savez_compressed(
        "SynCANCleanLabeled.npz",
        X_train=X_train_w,
        y_train=y_train_w,
        X_val=X_val_w,
        y_val=y_val_w
    )

if __name__ == "__main__":
    main()

MemoryError: Unable to allocate 312. MiB for an array with shape (40914620, 1) and data type int64

KeyboardInterrupt: 

NameError: name 'X_train' is not defined

In [ ]:

# 7. Simple feedforward NN (like INDRA notebook style)
# model = Sequential([
#     Dense(128, activation="relu", input_shape=(X_train.shape[1],)),
#     Dropout(0.3),
#     Dense(64, activation="relu"),
#     Dropout(0.3),
#     Dense(1, activation="sigmoid")
# ])

# model.compile(optimizer=Adam(learning_rate=1e-3),
#               loss="binary_crossentropy",
#               metrics=["accuracy"])


In [ ]:

# 8. Train
# history = model.fit(X_train, y_train,
#                     validation_data=(X_val, y_val),
#                     epochs=10,
#                     batch_size=256)


In [ ]:

# # 9. Evaluate
# loss, acc = model.evaluate(X_val, y_val)
# print(f"Validation accuracy: {acc:.3f}")